# 3. Geographic Health Equity Analysis
## Regional Healthcare Access and Disparity Intelligence Pipeline

Strategy: LIMIT 1000 on BigQuery reads -> Local CSV -> Pandas -> Spark

## Available Tables:
- **OMOP**: 24 tables
- **Medicare**: 6 tables
- **Dual Enrollment**: 1 table (SDOH)
- **CMS Codes**: 3 tables

## Pipeline Architecture:
- **Bronze**: 11 tables (raw geographic & clinical data)
- **Silver**: 6 intermediate layers (regional aggregations)
- **Gold**: 15 vertical layers -> 4 final metrics

## Final Metrics:
1. **Health Equity Index by Region** - Comprehensive access + outcome integration
2. **Care Desert Score** - Medical service availability index
3. **Social Determinants Impact** - SDOH influence measurement
4. **Geographic Disparity Magnitude** - Regional inequality quantification

In [1]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import *
import os
from datetime import datetime
import networkx as nx
import json

In [2]:
print("Initializing Spark...")
spark = SparkSession.builder \
    .appName("CMS_GeographicHealthEquity") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Initializing Spark...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 22:56:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1
Spark UI: http://mac:4043


25/12/02 22:56:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/02 22:56:14 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/12/02 22:56:14 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:
PROJECT_ID = "opportune-ruler-447319-b3"
LIMIT = 1000
LOCAL_DATA_DIR = "./3_data"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

DATASETS = {
    "omop": "bigquery-public-data.cms_synthetic_patient_data_omop",
    "medicare": "bigquery-public-data.cms_medicare",
    "dual": "bigquery-public-data.sdoh_cms_dual_eligible_enrollment",
    "hcpcs": "bigquery-public-data.cms_codes"
}

print(f"Config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Limit: {LIMIT} rows per table")
print(f"  Data dir: {LOCAL_DATA_DIR}")

Config:
  Project: opportune-ruler-447319-b3
  Limit: 1000 rows per table
  Data dir: ./3_data


# STEP 1: Download from BigQuery (LIMIT 1000)

In [4]:
def download_table(client, dataset_key, table_name, limit=1000):
    """Download table from BigQuery to local CSV"""
    try:
        source = DATASETS[dataset_key]
        query = f"SELECT * FROM `{source}.{table_name}` LIMIT {limit}"
        df = client.query(query).to_dataframe()
        
        output_file = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"  ✓ {dataset_key}.{table_name}: {len(df)} rows -> {output_file}")
        return True
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return False

In [5]:
print("\n" + "="*60)
print("DOWNLOADING TABLES")
print("="*60)

client = bigquery.Client(project=PROJECT_ID)

tables_to_download = [
    ("omop", "person"),
    ("omop", "location"),
    ("omop", "care_site"),
    ("omop", "provider"),
    ("omop", "condition_occurrence"),
    ("omop", "procedure_occurrence"),
    ("omop", "drug_exposure"),
    ("omop", "observation_period"),
    ("dual", "dual_eligible_enrollment_by_county_and_program"),
    ("medicare", "hospital_general_info"),
    ("medicare", "inpatient_charges_2011")
]

for dataset_key, table_name in tables_to_download:
    download_table(client, dataset_key, table_name, LIMIT)

print("\n✓ Download complete")


DOWNLOADING TABLES


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.person: 1000 rows -> ./data/omop_person.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.location: 1000 rows -> ./data/omop_location.csv


...

  ✓ dual.dual_eligible_enrollment_by_county_and_program: 1000 rows -> ./data/dual_dual_eligible_enrollment_by_county_and_program.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.hospital_general_info: 1000 rows -> ./data/medicare_hospital_general_info.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.inpatient_charges_2011: 1000 rows -> ./data/medicare_inpatient_charges_2011.csv

✓ Download complete


# STEP 2: Load CSV -> Pandas -> Spark (Bronze Layer)

In [4]:
def load_csv_to_spark(dataset_key, table_name):
    """Load CSV via Pandas then convert to Spark DataFrame"""
    try:
        csv_path = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        if not os.path.exists(csv_path):
            print(f"  ✗ {dataset_key}.{table_name}: File not found")
            return None
        
        pandas_df = pd.read_csv(csv_path)
        spark_df = spark.createDataFrame(pandas_df)
        print(f"  ✓ {dataset_key}.{table_name}: {spark_df.count()} rows")
        return spark_df
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return None

In [5]:
print("\n" + "="*60)
print("LOADING BRONZE LAYER")
print("="*60)

bronze_person = load_csv_to_spark("omop", "person")
bronze_location = load_csv_to_spark("omop", "location")
bronze_care_site = load_csv_to_spark("omop", "care_site")
bronze_provider = load_csv_to_spark("omop", "provider")
bronze_condition = load_csv_to_spark("omop", "condition_occurrence")
bronze_procedure = load_csv_to_spark("omop", "procedure_occurrence")
bronze_drug = load_csv_to_spark("omop", "drug_exposure")
bronze_obs_period = load_csv_to_spark("omop", "observation_period")
bronze_dual_enroll = load_csv_to_spark("dual", "dual_eligible_enrollment_by_county_and_program")
bronze_hospital_info = load_csv_to_spark("medicare", "hospital_general_info")
bronze_inpatient = load_csv_to_spark("medicare", "inpatient_charges_2011")

print("\n✓ Bronze Layer loaded (11 tables)")


LOADING BRONZE LAYER


  ✓ omop.person: 1000 rows
  ✓ omop.location: 1000 rows
  ✓ omop.care_site: 1000 rows
  ✓ omop.provider: 1000 rows
  ✓ omop.condition_occurrence: 1000 rows
  ✓ omop.procedure_occurrence: 1000 rows
  ✓ omop.drug_exposure: 1000 rows
  ✓ omop.observation_period: 1000 rows
  ✓ dual.dual_eligible_enrollment_by_county_and_program: 1000 rows
  ✗ medicare.hospital_general_info: [CANNOT_MERGE_TYPE] Can not merge type `BooleanType` and `DoubleType`.
  ✓ medicare.inpatient_charges_2011: 1000 rows

✓ Bronze Layer loaded (11 tables)


# STEP 3: Build SILVER Layer (6 Regional Aggregations)

In [6]:
print("\n" + "="*60)
print("SILVER 1/6: Regional Demographics Profile")
print("="*60)

silver_regional_demographics = bronze_person \
    .join(bronze_location, bronze_person.location_id == bronze_location.location_id, "left") \
    .groupBy("state", "zip", "county") \
    .agg(
        F.count("*").alias("population_count"),
        F.avg("year_of_birth").alias("avg_birth_year"),
        F.countDistinct("gender_concept_id").alias("gender_diversity"),
        F.countDistinct("race_concept_id").alias("race_diversity"),
        F.countDistinct("ethnicity_concept_id").alias("ethnicity_diversity")
    ) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_regional_demographics.count()}")
silver_regional_demographics.show(5, truncate=False)


SILVER 1/6: Regional Demographics Profile
  Regions: 191
+-----+---+------+----------------+------------------+----------------+--------------+-------------------+--------------------------+
|state|zip|county|population_count|avg_birth_year    |gender_diversity|race_diversity|ethnicity_diversity|processed_at              |
+-----+---+------+----------------+------------------+----------------+--------------+-------------------+--------------------------+
|FL   |NaN|10450 |1               |1917.0            |1               |1             |1                  |2025-12-02 22:56:27.985329|
|AZ   |NaN|3120  |3               |1934.0            |1               |1             |1                  |2025-12-02 22:56:27.985329|
|FL   |NaN|10570 |3               |1915.3333333333333|1               |1             |1                  |2025-12-02 22:56:27.985329|
|CA   |NaN|5430  |2               |1947.0            |1               |1             |1                  |2025-12-02 22:56:27.985329|
|IL 

In [7]:
print("\n" + "="*60)
print("SILVER 2/6: Provider Density by Region")
print("="*60)

provider_with_location = bronze_provider \
    .join(bronze_care_site, "care_site_id", "left") \
    .join(bronze_location, bronze_care_site.location_id == bronze_location.location_id, "left")

silver_provider_density = provider_with_location \
    .groupBy("state", "zip", "county") \
    .agg(
        F.count("*").alias("total_providers"),
        F.countDistinct("specialty_concept_id").alias("specialty_diversity"),
        F.countDistinct("care_site_id").alias("care_sites_count"),
        F.avg("year_of_birth").alias("avg_provider_age")
    ) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_provider_density.count()}")
silver_provider_density.show(5, truncate=False)


SILVER 2/6: Provider Density by Region
  Regions: 1
+-----+----+------+---------------+-------------------+----------------+----------------+--------------------------+
|state|zip |county|total_providers|specialty_diversity|care_sites_count|avg_provider_age|processed_at              |
+-----+----+------+---------------+-------------------+----------------+----------------+--------------------------+
|NULL |NULL|NULL  |1000           |1                  |162             |NaN             |2025-12-02 22:56:29.313729|
+-----+----+------+---------------+-------------------+----------------+----------------+--------------------------+



In [8]:
print("\n" + "="*60)
print("SILVER 3/6: Regional Service Utilization")
print("="*60)

person_location = bronze_person \
    .join(bronze_location, "location_id", "left") \
    .select("person_id", "state", "zip", "county")

condition_regional = bronze_condition \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county") \
    .agg(F.count("*").alias("condition_events"))

procedure_regional = bronze_procedure \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county") \
    .agg(F.count("*").alias("procedure_events"))

drug_regional = bronze_drug \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county") \
    .agg(F.count("*").alias("drug_events"))

silver_service_utilization = condition_regional \
    .join(procedure_regional, ["state", "zip", "county"], "outer") \
    .join(drug_regional, ["state", "zip", "county"], "outer") \
    .fillna(0) \
    .withColumn("total_events", 
        F.col("condition_events") + F.col("procedure_events") + F.col("drug_events")) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_service_utilization.count()}")
silver_service_utilization.show(5, truncate=False)


SILVER 3/6: Regional Service Utilization
  Regions: 3
+-----+---+------+----------------+----------------+-----------+------------+--------------------------+
|state|zip|county|condition_events|procedure_events|drug_events|total_events|processed_at              |
+-----+---+------+----------------+----------------+-----------+------------+--------------------------+
|NULL |0.0|0     |1000            |0               |0          |1000        |2025-12-02 22:56:30.487562|
|NULL |0.0|0     |0               |1000            |0          |1000        |2025-12-02 22:56:30.487562|
|NULL |0.0|0     |0               |0               |1000       |1000        |2025-12-02 22:56:30.487562|
+-----+---+------+----------------+----------------+-----------+------------+--------------------------+



In [9]:
print("\n" + "="*60)
print("SILVER 4/6: Regional Health Outcomes")
print("="*60)

chronic_conditions = bronze_condition \
    .join(person_location, "person_id", "left") \
    .groupBy("state", "zip", "county", "person_id") \
    .agg(F.countDistinct("condition_concept_id").alias("condition_count"))

silver_health_outcomes = chronic_conditions \
    .groupBy("state", "zip", "county") \
    .agg(
        F.avg("condition_count").alias("avg_conditions_per_patient"),
        F.max("condition_count").alias("max_conditions_per_patient"),
        F.count(F.when(F.col("condition_count") >= 3, 1)).alias("complex_patients_count")
    ) \
    .withColumn("processed_at", F.current_timestamp())

print(f"  Regions: {silver_health_outcomes.count()}")
silver_health_outcomes.show(5, truncate=False)


SILVER 4/6: Regional Health Outcomes
  Regions: 1
+-----+----+------+--------------------------+--------------------------+----------------------+--------------------------+
|state|zip |county|avg_conditions_per_patient|max_conditions_per_patient|complex_patients_count|processed_at              |
+-----+----+------+--------------------------+--------------------------+----------------------+--------------------------+
|NULL |NULL|NULL  |1.0                       |1                         |0                     |2025-12-02 22:56:31.500037|
+-----+----+------+--------------------------+--------------------------+----------------------+--------------------------+



In [10]:
print("\n" + "="*60)
print("SILVER 5/6: SDOH (Social Determinants of Health) Indicators")
print("="*60)

if bronze_dual_enroll is not None:
    silver_sdoh_indicators = bronze_dual_enroll \
        .groupBy("State_Abbr", "County_Name") \
        .agg(
            F.sum("Public_Total").alias("total_dual_eligible"),
            F.sum("QMB_plus_Full").alias("qmb_plus_full_count"),
            F.sum("QMB_Only").alias("qmb_only_count"),
            F.sum("SLMB_only").alias("slmb_only_count"),
            F.sum("SLMB_plus_Full").alias("slmb_plus_full_count"),
            F.sum("QDWI").alias("qdwi_count"),
            F.sum("QI").alias("qi_count"),
            F.sum("Other_full").alias("other_dual_count")
        ) \
        .withColumn("dual_eligible_ratio",
            F.col("qmb_plus_full_count") / (F.col("total_dual_eligible") + 1)) \
        .withColumn("processed_at", F.current_timestamp())
    
    print(f"  Regions: {silver_sdoh_indicators.count()}")
    silver_sdoh_indicators.show(5, truncate=False)
else:
    print("  ✗ Dual enrollment data not available")
    silver_sdoh_indicators = None


SILVER 5/6: SDOH (Social Determinants of Health) Indicators
  Regions: 125
+----------+-----------+-------------------+-------------------+--------------+---------------+--------------------+----------+--------+----------------+-------------------+--------------------------+
|State_Abbr|County_Name|total_dual_eligible|qmb_plus_full_count|qmb_only_count|slmb_only_count|slmb_plus_full_count|qdwi_count|qi_count|other_dual_count|dual_eligible_ratio|processed_at              |
+----------+-----------+-------------------+-------------------+--------------+---------------+--------------------+----------+--------+----------------+-------------------+--------------------------+
|AL        |BULLOCK    |4172               |1500               |1350          |599            |NaN                 |0.0       |359     |364.0           |0.35945363048166784|2025-12-02 22:56:32.057048|
|AL        |Autauga    |15662              |4521               |5360          |2618           |259.0               |0.0 

In [11]:
print("\n" + "="*60)
print("SILVER 6/6: Facility Access and Quality")
print("="*60)

if bronze_hospital_info is not None:
    silver_facility_access = bronze_hospital_info \
        .groupBy("state", "zip_code", "county_name") \
        .agg(
            F.count("*").alias("hospital_count"),
            F.countDistinct("hospital_type").alias("hospital_type_diversity"),
            F.countDistinct("hospital_ownership").alias("ownership_diversity"),
            F.avg("hospital_overall_rating").alias("avg_hospital_rating")
        ) \
        .withColumn("processed_at", F.current_timestamp())
    
    print(f"  Regions: {silver_facility_access.count()}")
    silver_facility_access.show(5, truncate=False)
else:
    print("  ✗ Hospital info data not available")
    silver_facility_access = None

print("\n✓ Silver Layer complete (6 tables)")


SILVER 6/6: Facility Access and Quality
  ✗ Hospital info data not available

✓ Silver Layer complete (6 tables)


# STEP 4: Build GOLD Layer (15 Vertical Transformations)

In [12]:
print("\n" + "="*60)
print("GOLD 1/15: Population Health Index")
print("="*60)

gold_population_health_index = silver_regional_demographics \
    .join(silver_health_outcomes, ["state", "zip", "county"], "left") \
    .withColumn("diversity_score",
        (F.col("gender_diversity") + F.col("race_diversity") + F.col("ethnicity_diversity")) / 3) \
    .withColumn("health_burden_score",
        F.col("avg_conditions_per_patient") * F.col("complex_patients_count")) \
    .select(
        "state", "zip", "county",
        "population_count",
        "diversity_score",
        "health_burden_score",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_population_health_index.count()}")
gold_population_health_index.show(5, truncate=False)


GOLD 1/15: Population Health Index
  Regions: 191
+-----+---+------+----------------+---------------+-------------------+-------------------------+
|state|zip|county|population_count|diversity_score|health_burden_score|processed_at             |
+-----+---+------+----------------+---------------+-------------------+-------------------------+
|FL   |NaN|10450 |1               |1.0            |NULL               |2025-12-02 22:56:32.47454|
|AZ   |NaN|3120  |3               |1.0            |NULL               |2025-12-02 22:56:32.47454|
|FL   |NaN|10570 |3               |1.0            |NULL               |2025-12-02 22:56:32.47454|
|CA   |NaN|5430  |2               |1.0            |NULL               |2025-12-02 22:56:32.47454|
|IL   |NaN|14989 |2               |1.0            |NULL               |2025-12-02 22:56:32.47454|
+-----+---+------+----------------+---------------+-------------------+-------------------------+
only showing top 5 rows



In [13]:
print("\n" + "="*60)
print("GOLD 2/15: Provider-to-Population Access Ratio")
print("="*60)

gold_access_ratio = silver_regional_demographics \
    .join(silver_provider_density, ["state", "zip", "county"], "left") \
    .withColumn("providers_per_1000",
        (F.col("total_providers") / F.col("population_count")) * 1000) \
    .withColumn("specialties_per_1000",
        (F.col("specialty_diversity") / F.col("population_count")) * 1000) \
    .withColumn("care_sites_per_1000",
        (F.col("care_sites_count") / F.col("population_count")) * 1000) \
    .select(
        "state", "zip", "county",
        "providers_per_1000",
        "specialties_per_1000",
        "care_sites_per_1000",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_access_ratio.count()}")
gold_access_ratio.show(5, truncate=False)


GOLD 2/15: Provider-to-Population Access Ratio
  Regions: 191
+-----+---+------+------------------+--------------------+-------------------+--------------------------+
|state|zip|county|providers_per_1000|specialties_per_1000|care_sites_per_1000|processed_at              |
+-----+---+------+------------------+--------------------+-------------------+--------------------------+
|CT   |NaN|7040  |NULL              |NULL                |NULL               |2025-12-02 22:56:33.306688|
|AZ   |NaN|3060  |NULL              |NULL                |NULL               |2025-12-02 22:56:33.306688|
|CO   |NaN|6000  |NULL              |NULL                |NULL               |2025-12-02 22:56:33.306688|
|FL   |NaN|10450 |NULL              |NULL                |NULL               |2025-12-02 22:56:33.306688|
|54   |NaN|54070 |NULL              |NULL                |NULL               |2025-12-02 22:56:33.306688|
+-----+---+------+------------------+--------------------+-------------------+-----------

In [14]:
print("\n" + "="*60)
print("GOLD 3/15: Service Utilization Intensity")
print("="*60)

gold_utilization_intensity = silver_service_utilization \
    .join(silver_regional_demographics, ["state", "zip", "county"], "left") \
    .withColumn("events_per_capita",
        F.col("total_events") / F.col("population_count")) \
    .withColumn("condition_intensity",
        F.col("condition_events") / F.col("population_count")) \
    .withColumn("procedure_intensity",
        F.col("procedure_events") / F.col("population_count")) \
    .withColumn("drug_intensity",
        F.col("drug_events") / F.col("population_count")) \
    .select(
        "state", "zip", "county",
        "events_per_capita",
        "condition_intensity",
        "procedure_intensity",
        "drug_intensity",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_utilization_intensity.count()}")
gold_utilization_intensity.show(5, truncate=False)


GOLD 3/15: Service Utilization Intensity
  Regions: 3
+-----+---+------+-----------------+-------------------+-------------------+--------------+--------------------------+
|state|zip|county|events_per_capita|condition_intensity|procedure_intensity|drug_intensity|processed_at              |
+-----+---+------+-----------------+-------------------+-------------------+--------------+--------------------------+
|NULL |0.0|0     |NULL             |NULL               |NULL               |NULL          |2025-12-02 22:56:34.360888|
|NULL |0.0|0     |NULL             |NULL               |NULL               |NULL          |2025-12-02 22:56:34.360888|
|NULL |0.0|0     |NULL             |NULL               |NULL               |NULL          |2025-12-02 22:56:34.360888|
+-----+---+------+-----------------+-------------------+-------------------+--------------+--------------------------+



In [15]:
print("\n" + "="*60)
print("GOLD 4/15: Population Vulnerability Score")
print("="*60)

if silver_sdoh_indicators is not None:
    gold_vulnerability_score = silver_sdoh_indicators \
        .withColumn("vulnerability_index",
            (F.col("dual_eligible_ratio") * 0.4 +
             (F.col("qmb_only_count") / F.col("total_dual_eligible")) * 0.3 +
             (F.col("other_dual_count") / F.col("total_dual_eligible")) * 0.3)) \
        .select(
            F.col("State_Abbr").alias("state"),
            F.col("County_Name").alias("county"),
            "vulnerability_index",
            "total_dual_eligible",
            "dual_eligible_ratio",
            F.current_timestamp().alias("processed_at")
        )
    
    print(f"  Regions: {gold_vulnerability_score.count()}")
    gold_vulnerability_score.show(5, truncate=False)
else:
    print("  ✗ Skipping - SDOH data not available")
    gold_vulnerability_score = None


GOLD 4/15: Population Vulnerability Score
  Regions: 125
+-----+-------+-------------------+-------------------+-------------------+-------------------------+
|state|county |vulnerability_index|total_dual_eligible|dual_eligible_ratio|processed_at             |
+-----+-------+-------------------+-------------------+-------------------+-------------------------+
|AL   |BULLOCK|0.26703169188585985|4172               |0.35945363048166784|2025-12-02 22:56:35.20107|
|AL   |Autauga|0.23741441343321157|15662              |0.2886420226010343 |2025-12-02 22:56:35.20107|
|AL   |Barbour|0.26044506936138034|14601              |0.35385563621421723|2025-12-02 22:56:35.20107|
|AL   |AUTAUGA|0.23450715448801351|13875              |0.28307869703084465|2025-12-02 22:56:35.20107|
|AL   |Bibb   |0.2603978336248962 |9877               |0.37649321725045554|2025-12-02 22:56:35.20107|
+-----+-------+-------------------+-------------------+-------------------+-------------------------+
only showing top 5 rows


In [16]:
print("\n" + "="*60)
print("GOLD 5/15: Quality-Adjusted Access Score")
print("="*60)

if silver_facility_access is not None:
    gold_quality_access = silver_facility_access \
        .join(silver_regional_demographics, 
              [silver_facility_access.state == silver_regional_demographics.state,
               silver_facility_access.zip_code == silver_regional_demographics.zip], "left") \
        .withColumn("quality_adjusted_access",
            (F.col("hospital_count") / F.col("population_count") * 10000) * 
            F.coalesce(F.col("avg_hospital_rating"), F.lit(3))) \
        .select(
            silver_facility_access.state,
            silver_facility_access.zip_code.alias("zip"),
            silver_facility_access.county_name.alias("county"),
            "quality_adjusted_access",
            "hospital_count",
            "avg_hospital_rating",
            F.current_timestamp().alias("processed_at")
        )
    
    print(f"  Regions: {gold_quality_access.count()}")
    gold_quality_access.show(5, truncate=False)
else:
    print("  ✗ Skipping - Facility data not available")
    gold_quality_access = None


GOLD 5/15: Quality-Adjusted Access Score
  ✗ Skipping - Facility data not available


In [17]:
print("\n" + "="*60)
print("GOLD 6/15: Healthcare Service Diversity Index")
print("="*60)

gold_service_diversity = silver_provider_density \
    .withColumn("service_diversity_index",
        (F.col("specialty_diversity") * 0.6 + F.col("care_sites_count") * 0.4) / 10) \
    .select(
        "state", "zip", "county",
        "service_diversity_index",
        "specialty_diversity",
        "care_sites_count",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_service_diversity.count()}")
gold_service_diversity.show(5, truncate=False)


GOLD 6/15: Healthcare Service Diversity Index
  Regions: 1
+-----+----+------+-----------------------+-------------------+----------------+--------------------------+
|state|zip |county|service_diversity_index|specialty_diversity|care_sites_count|processed_at              |
+-----+----+------+-----------------------+-------------------+----------------+--------------------------+
|NULL |NULL|NULL  |6.539999999999999      |1                  |162             |2025-12-02 22:56:35.713731|
+-----+----+------+-----------------------+-------------------+----------------+--------------------------+



In [18]:
print("\n" + "="*60)
print("GOLD 7/15: Multi-Dimensional Care Accessibility")
print("="*60)

gold_care_accessibility = gold_access_ratio \
    .join(gold_service_diversity, ["state", "zip", "county"], "left") \
    .withColumn("accessibility_composite",
        (F.coalesce(F.col("providers_per_1000"), F.lit(0)) * 0.4 +
         F.coalesce(F.col("specialties_per_1000"), F.lit(0)) * 10 * 0.3 +
         F.coalesce(F.col("service_diversity_index"), F.lit(0)) * 0.3)) \
    .select(
        "state", "zip", "county",
        "accessibility_composite",
        "providers_per_1000",
        "specialties_per_1000",
        "service_diversity_index",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_care_accessibility.count()}")
gold_care_accessibility.show(5, truncate=False)


GOLD 7/15: Multi-Dimensional Care Accessibility
  Regions: 191
+-----+---+------+-----------------------+------------------+--------------------+-----------------------+--------------------------+
|state|zip|county|accessibility_composite|providers_per_1000|specialties_per_1000|service_diversity_index|processed_at              |
+-----+---+------+-----------------------+------------------+--------------------+-----------------------+--------------------------+
|CT   |NaN|7040  |0.0                    |NULL              |NULL                |NULL                   |2025-12-02 22:56:36.316171|
|AZ   |NaN|3060  |0.0                    |NULL              |NULL                |NULL                   |2025-12-02 22:56:36.316171|
|CO   |NaN|6000  |0.0                    |NULL              |NULL                |NULL                   |2025-12-02 22:56:36.316171|
|FL   |NaN|10450 |0.0                    |NULL              |NULL                |NULL                   |2025-12-02 22:56:36.316171

In [19]:
print("\n" + "="*60)
print("GOLD 8/15: Healthcare Need vs Supply Gap")
print("="*60)

gold_need_supply_gap = gold_population_health_index \
    .join(gold_care_accessibility, ["state", "zip", "county"], "left") \
    .withColumn("need_score",
        F.col("health_burden_score") / (F.col("diversity_score") + 1)) \
    .withColumn("supply_score",
        F.coalesce(F.col("accessibility_composite"), F.lit(0))) \
    .withColumn("gap_magnitude",
        F.col("need_score") - F.col("supply_score")) \
    .select(
        "state", "zip", "county",
        "need_score",
        "supply_score",
        "gap_magnitude",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_need_supply_gap.count()}")
gold_need_supply_gap.show(5, truncate=False)


GOLD 8/15: Healthcare Need vs Supply Gap
  Regions: 191
+-----+---+------+----------+------------+-------------+--------------------------+
|state|zip|county|need_score|supply_score|gap_magnitude|processed_at              |
+-----+---+------+----------+------------+-------------+--------------------------+
|FL   |NaN|10450 |NULL      |0.0         |NULL         |2025-12-02 22:56:37.029108|
|FL   |NaN|10570 |NULL      |0.0         |NULL         |2025-12-02 22:56:37.029108|
|CA   |NaN|5430  |NULL      |0.0         |NULL         |2025-12-02 22:56:37.029108|
|IL   |NaN|14989 |NULL      |0.0         |NULL         |2025-12-02 22:56:37.029108|
|CT   |NaN|7020  |NULL      |0.0         |NULL         |2025-12-02 22:56:37.029108|
+-----+---+------+----------+------------+-------------+--------------------------+
only showing top 5 rows



In [20]:
print("\n" + "="*60)
print("GOLD 9/15: Underserved Area Classification")
print("="*60)

window_spec = W.partitionBy()

gold_underserved_flag = gold_need_supply_gap \
    .withColumn("gap_percentile",
        F.percent_rank().over(window_spec.orderBy(F.col("gap_magnitude").desc()))) \
    .withColumn("underserved_severity",
        F.when(F.col("gap_percentile") < 0.1, "Critical")
         .when(F.col("gap_percentile") < 0.3, "High")
         .when(F.col("gap_percentile") < 0.6, "Moderate")
         .otherwise("Low")) \
    .select(
        "state", "zip", "county",
        "gap_magnitude",
        "gap_percentile",
        "underserved_severity",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_underserved_flag.count()}")
gold_underserved_flag.show(5, truncate=False)


GOLD 9/15: Underserved Area Classification
  Regions: 191


25/12/02 22:56:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+---+------+-------------+--------------+--------------------+--------------------------+
|state|zip|county|gap_magnitude|gap_percentile|underserved_severity|processed_at              |
+-----+---+------+-------------+--------------+--------------------+--------------------------+
|FL   |NaN|10450 |NULL         |0.0           |Critical            |2025-12-02 22:56:38.190376|
|FL   |NaN|10570 |NULL         |0.0           |Critical            |2025-12-02 22:56:38.190376|
|CA   |NaN|5430  |NULL         |0.0           |Critical            |2025-12-02 22:56:38.190376|
|IL   |NaN|14989 |NULL         |0.0           |Critical            |2025-12-02 22:56:38.190376|
|CT   |NaN|7020  |NULL         |0.0           |Critical            |2025-12-02 22:56:38.190376|
+-----+---+------+-------------+--------------+--------------------+--------------------------+
only showing top 5 rows



25/12/02 22:56:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [21]:
print("\n" + "="*60)
print("GOLD 10/15: Utilization Efficiency Score")
print("="*60)

gold_utilization_efficiency = gold_utilization_intensity \
    .join(gold_care_accessibility, ["state", "zip", "county"], "left") \
    .withColumn("efficiency_ratio",
        F.col("events_per_capita") / 
        (F.coalesce(F.col("accessibility_composite"), F.lit(1)) + 0.01)) \
    .withColumn("normalized_efficiency",
        F.when(F.col("efficiency_ratio") > 2, F.lit("Over-utilized"))
         .when(F.col("efficiency_ratio") > 0.5, F.lit("Balanced"))
         .otherwise("Under-utilized")) \
    .select(
        "state", "zip", "county",
        "efficiency_ratio",
        "normalized_efficiency",
        "events_per_capita",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_utilization_efficiency.count()}")
gold_utilization_efficiency.show(5, truncate=False)


GOLD 10/15: Utilization Efficiency Score
  Regions: 3


+-----+---+------+----------------+---------------------+-----------------+--------------------------+
|state|zip|county|efficiency_ratio|normalized_efficiency|events_per_capita|processed_at              |
+-----+---+------+----------------+---------------------+-----------------+--------------------------+
|NULL |0.0|0     |NULL            |Under-utilized       |NULL             |2025-12-02 22:56:39.814303|
|NULL |0.0|0     |NULL            |Under-utilized       |NULL             |2025-12-02 22:56:39.814303|
|NULL |0.0|0     |NULL            |Under-utilized       |NULL             |2025-12-02 22:56:39.814303|
+-----+---+------+----------------+---------------------+-----------------+--------------------------+



In [22]:
print("\n" + "="*60)
print("GOLD 11/15: Regional Inequity Magnitude")
print("="*60)

state_avg = gold_care_accessibility \
    .groupBy("state") \
    .agg(F.avg("accessibility_composite").alias("state_avg_access"))

gold_regional_inequity = gold_care_accessibility \
    .join(state_avg, "state", "left") \
    .withColumn("access_deviation",
        F.col("accessibility_composite") - F.col("state_avg_access")) \
    .withColumn("inequity_magnitude",
        F.abs(F.col("access_deviation")) / (F.col("state_avg_access") + 0.01)) \
    .select(
        "state", "zip", "county",
        "access_deviation",
        "inequity_magnitude",
        "state_avg_access",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_regional_inequity.count()}")
gold_regional_inequity.show(5, truncate=False)


GOLD 11/15: Regional Inequity Magnitude
  Regions: 191
+-----+---+------+----------------+------------------+----------------+--------------------------+
|state|zip|county|access_deviation|inequity_magnitude|state_avg_access|processed_at              |
+-----+---+------+----------------+------------------+----------------+--------------------------+
|CT   |NaN|7040  |0.0             |0.0               |0.0             |2025-12-02 22:56:41.206408|
|AZ   |NaN|3060  |0.0             |0.0               |0.0             |2025-12-02 22:56:41.206408|
|CO   |NaN|6000  |0.0             |0.0               |0.0             |2025-12-02 22:56:41.206408|
|FL   |NaN|10450 |0.0             |0.0               |0.0             |2025-12-02 22:56:41.206408|
|54   |NaN|54070 |0.0             |0.0               |0.0             |2025-12-02 22:56:41.206408|
+-----+---+------+----------------+------------------+----------------+--------------------------+
only showing top 5 rows



In [23]:
print("\n" + "="*60)
print("GOLD 12/15: Compound Disadvantage Index")
print("="*60)

base_disadvantage = gold_underserved_flag \
    .join(gold_utilization_efficiency, ["state", "zip", "county"], "left")

if gold_vulnerability_score is not None:
    gold_compound_disadvantage = base_disadvantage \
        .join(gold_vulnerability_score,
              [base_disadvantage.state == gold_vulnerability_score.state,
               base_disadvantage.county == gold_vulnerability_score.county], "left") \
        .withColumn("compound_index",
            (F.col("gap_percentile") * 0.4 +
             F.when(F.col("normalized_efficiency") == "Under-utilized", 0.3).otherwise(0) +
             F.coalesce(F.col("vulnerability_index"), F.lit(0)) * 0.3)) \
        .select(
            base_disadvantage.state,
            base_disadvantage.zip,
            base_disadvantage.county,
            "compound_index",
            "underserved_severity",
            "normalized_efficiency",
            F.current_timestamp().alias("processed_at")
        )
else:
    gold_compound_disadvantage = base_disadvantage \
        .withColumn("compound_index",
            (F.col("gap_percentile") * 0.6 +
             F.when(F.col("normalized_efficiency") == "Under-utilized", 0.4).otherwise(0))) \
        .select(
            "state", "zip", "county",
            "compound_index",
            "underserved_severity",
            "normalized_efficiency",
            F.current_timestamp().alias("processed_at")
        )

print(f"  Regions: {gold_compound_disadvantage.count()}")
gold_compound_disadvantage.show(5, truncate=False)


GOLD 12/15: Compound Disadvantage Index
  Regions: 191


25/12/02 22:56:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+---+------+--------------+--------------------+---------------------+--------------------------+
|state|zip|county|compound_index|underserved_severity|normalized_efficiency|processed_at              |
+-----+---+------+--------------+--------------------+---------------------+--------------------------+
|FL   |NaN|10450 |0.0           |Critical            |NULL                 |2025-12-02 22:56:42.623448|
|FL   |NaN|10570 |0.0           |Critical            |NULL                 |2025-12-02 22:56:42.623448|
|CA   |NaN|5430  |0.0           |Critical            |NULL                 |2025-12-02 22:56:42.623448|
|IL   |NaN|14989 |0.0           |Critical            |NULL                 |2025-12-02 22:56:42.623448|
|CT   |NaN|7020  |0.0           |Critical            |NULL                 |2025-12-02 22:56:42.623448|
+-----+---+------+--------------+--------------------+---------------------+--------------------------+
only showing top 5 rows



In [24]:
print("\n" + "="*60)
print("GOLD 13/15: Temporal Stability Assessment")
print("="*60)

gold_temporal_stability = gold_utilization_intensity \
    .withColumn("service_variance",
        F.pow(F.col("condition_intensity") - F.col("drug_intensity"), 2) +
        F.pow(F.col("procedure_intensity") - F.col("drug_intensity"), 2)) \
    .withColumn("stability_score",
        F.lit(1) / (F.col("service_variance") + 0.01)) \
    .select(
        "state", "zip", "county",
        "stability_score",
        "service_variance",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_temporal_stability.count()}")
gold_temporal_stability.show(5, truncate=False)


GOLD 13/15: Temporal Stability Assessment
  Regions: 3
+-----+---+------+---------------+----------------+--------------------------+
|state|zip|county|stability_score|service_variance|processed_at              |
+-----+---+------+---------------+----------------+--------------------------+
|NULL |0.0|0     |NULL           |NULL            |2025-12-02 22:56:44.628953|
|NULL |0.0|0     |NULL           |NULL            |2025-12-02 22:56:44.628953|
|NULL |0.0|0     |NULL           |NULL            |2025-12-02 22:56:44.628953|
+-----+---+------+---------------+----------------+--------------------------+



In [25]:
print("\n" + "="*60)
print("GOLD 14/15: Integrated Regional Ranking")
print("="*60)

window_national = W.partitionBy()

gold_integrated_ranking = gold_care_accessibility \
    .join(gold_need_supply_gap, ["state", "zip", "county"], "left") \
    .join(gold_compound_disadvantage, ["state", "zip", "county"], "left") \
    .withColumn("composite_score",
        F.coalesce(F.col("accessibility_composite"), F.lit(0)) * 0.4 -
        F.col("gap_magnitude") * 0.3 -
        F.coalesce(F.col("compound_index"), F.lit(0)) * 0.3) \
    .withColumn("national_percentile",
        F.percent_rank().over(window_national.orderBy(F.col("composite_score").desc()))) \
    .withColumn("tier",
        F.when(F.col("national_percentile") < 0.2, "Top 20%")
         .when(F.col("national_percentile") < 0.5, "Above Average")
         .when(F.col("national_percentile") < 0.8, "Below Average")
         .otherwise("Bottom 20%")) \
    .select(
        "state", "zip", "county",
        "composite_score",
        "national_percentile",
        "tier",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_integrated_ranking.count()}")
gold_integrated_ranking.show(5, truncate=False)


GOLD 14/15: Integrated Regional Ranking
  Regions: 191


25/12/02 22:56:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+----+------+---------------+-------------------+-------+--------------------------+
|state|zip |county|composite_score|national_percentile|tier   |processed_at              |
+-----+----+------+---------------+-------------------+-------+--------------------------+
|NULL |NULL|NULL  |NULL           |0.0                |Top 20%|2025-12-02 22:56:46.289854|
|54   |NaN |54000 |NULL           |0.0                |Top 20%|2025-12-02 22:56:46.289854|
|54   |NaN |54010 |NULL           |0.0                |Top 20%|2025-12-02 22:56:46.289854|
|54   |NaN |54040 |NULL           |0.0                |Top 20%|2025-12-02 22:56:46.289854|
|54   |NaN |54070 |NULL           |0.0                |Top 20%|2025-12-02 22:56:46.289854|
+-----+----+------+---------------+-------------------+-------+--------------------------+
only showing top 5 rows



25/12/02 22:56:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [26]:
print("\n" + "="*60)
print("GOLD 15/15: Predictive Deterioration Risk")
print("="*60)

gold_predictive_risk = gold_need_supply_gap \
    .join(gold_temporal_stability, ["state", "zip", "county"], "left") \
    .join(gold_compound_disadvantage, ["state", "zip", "county"], "left") \
    .withColumn("risk_score",
        F.col("gap_magnitude") * 0.4 +
        (F.lit(1) - F.coalesce(F.col("stability_score"), F.lit(0.5))) * 0.3 +
        F.coalesce(F.col("compound_index"), F.lit(0)) * 0.3) \
    .withColumn("risk_category",
        F.when(F.col("risk_score") > 0.7, "High Risk")
         .when(F.col("risk_score") > 0.4, "Moderate Risk")
         .otherwise("Low Risk")) \
    .select(
        "state", "zip", "county",
        "risk_score",
        "risk_category",
        F.current_timestamp().alias("processed_at")
    )

print(f"  Regions: {gold_predictive_risk.count()}")
gold_predictive_risk.show(5, truncate=False)

print("\n✓ Gold Layer complete (15 transformations)")


GOLD 15/15: Predictive Deterioration Risk
  Regions: 191


25/12/02 22:56:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+----+------+----------+-------------+--------------------------+
|state|zip |county|risk_score|risk_category|processed_at              |
+-----+----+------+----------+-------------+--------------------------+
|NULL |NULL|NULL  |NULL      |Low Risk     |2025-12-02 22:56:48.963948|
|54   |NaN |54000 |NULL      |Low Risk     |2025-12-02 22:56:48.963948|
|54   |NaN |54010 |NULL      |Low Risk     |2025-12-02 22:56:48.963948|
|54   |NaN |54040 |NULL      |Low Risk     |2025-12-02 22:56:48.963948|
|54   |NaN |54070 |NULL      |Low Risk     |2025-12-02 22:56:48.963948|
+-----+----+------+----------+-------------+--------------------------+
only showing top 5 rows


✓ Gold Layer complete (15 transformations)


# STEP 5: Final Metrics (4 Top-Level KPIs)

In [27]:
print("\n" + "="*60)
print("METRIC 1/4: Health Equity Index by Region")
print("="*60)

metric_health_equity_index = gold_integrated_ranking \
    .join(gold_care_accessibility, ["state", "zip", "county"], "left") \
    .join(gold_population_health_index, ["state", "zip", "county"], "left") \
    .withColumn("equity_index",
        (F.col("composite_score") * 0.5 +
         (F.lit(1) - F.col("national_percentile")) * 0.3 +
         (F.lit(1) / (F.col("health_burden_score") + 1)) * 0.2) * 100) \
    .withColumn("equity_grade",
        F.when(F.col("equity_index") >= 80, "A")
         .when(F.col("equity_index") >= 60, "B")
         .when(F.col("equity_index") >= 40, "C")
         .when(F.col("equity_index") >= 20, "D")
         .otherwise("F")) \
    .select(
        "state", "zip", "county",
        F.round("equity_index", 2).alias("health_equity_index"),
        "equity_grade",
        "tier",
        F.current_timestamp().alias("calculated_at")
    )

print(f"  Total regions: {metric_health_equity_index.count()}")
metric_health_equity_index.orderBy(F.desc("health_equity_index")).show(10, truncate=False)

print("\nEquity Grade Distribution:")
metric_health_equity_index.groupBy("equity_grade").count().orderBy("equity_grade").show()


METRIC 1/4: Health Equity Index by Region


  Total regions: 191


25/12/02 22:56:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+----+------+-------------------+------------+-------+--------------------------+
|state|zip |county|health_equity_index|equity_grade|tier   |calculated_at             |
+-----+----+------+-------------------+------------+-------+--------------------------+
|NULL |NULL|NULL  |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54000 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54010 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54040 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54070 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54170 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54310 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54340 |NULL               |F           |Top 20%|2025-12-02 22:56:52.131939|
|54   |NaN |54999 |NULL         

25/12/02 22:56:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+------------+-----+
|equity_grade|count|
+------------+-----+
|           F|  191|
+------------+-----+



In [28]:
print("\n" + "="*60)
print("METRIC 2/4: Care Desert Score")
print("="*60)

metric_care_desert = gold_underserved_flag \
    .join(gold_access_ratio, ["state", "zip", "county"], "left") \
    .join(gold_service_diversity, ["state", "zip", "county"], "left") \
    .withColumn("desert_score",
        (F.col("gap_percentile") * 0.4 +
         (F.lit(1) / (F.coalesce(F.col("providers_per_1000"), F.lit(0.1)) + 0.1)) * 0.3 +
         (F.lit(1) / (F.coalesce(F.col("service_diversity_index"), F.lit(0.1)) + 0.1)) * 0.3) * 100) \
    .withColumn("desert_classification",
        F.when(F.col("desert_score") >= 80, "Severe Desert")
         .when(F.col("desert_score") >= 60, "Moderate Desert")
         .when(F.col("desert_score") >= 40, "Mild Desert")
         .otherwise("Adequate Coverage")) \
    .select(
        "state", "zip", "county",
        F.round("desert_score", 2).alias("care_desert_score"),
        "desert_classification",
        "underserved_severity",
        F.current_timestamp().alias("calculated_at")
    )

print(f"  Total regions: {metric_care_desert.count()}")
metric_care_desert.orderBy(F.desc("care_desert_score")).show(10, truncate=False)

print("\nDesert Classification Distribution:")
metric_care_desert.groupBy("desert_classification").count().orderBy("desert_classification").show()


METRIC 2/4: Care Desert Score
  Total regions: 191


25/12/02 22:56:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+---+------+-----------------+---------------------+--------------------+--------------------------+
|state|zip|county|care_desert_score|desert_classification|underserved_severity|calculated_at             |
+-----+---+------+-----------------+---------------------+--------------------+--------------------------+
|FL   |NaN|10450 |300.0            |Severe Desert        |Critical            |2025-12-02 22:56:56.355583|
|FL   |NaN|10570 |300.0            |Severe Desert        |Critical            |2025-12-02 22:56:56.355583|
|CA   |NaN|5430  |300.0            |Severe Desert        |Critical            |2025-12-02 22:56:56.355583|
|IL   |NaN|14989 |300.0            |Severe Desert        |Critical            |2025-12-02 22:56:56.355583|
|CT   |NaN|7020  |300.0            |Severe Desert        |Critical            |2025-12-02 22:56:56.355583|
|CO   |NaN|6500  |300.0            |Severe Desert        |Critical            |2025-12-02 22:56:56.355583|
|IN   |NaN|15170 |300.0            |S

25/12/02 22:56:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+---------------------+-----+
|desert_classification|count|
+---------------------+-----+
|        Severe Desert|  191|
+---------------------+-----+



25/12/02 22:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [29]:
print("\n" + "="*60)
print("METRIC 3/4: Social Determinants Impact")
print("="*60)

if gold_vulnerability_score is not None:
    metric_sdoh_impact = gold_vulnerability_score \
        .join(gold_compound_disadvantage,
              [gold_vulnerability_score.state == gold_compound_disadvantage.state,
               gold_vulnerability_score.county == gold_compound_disadvantage.county], "left") \
        .withColumn("sdoh_impact_score",
            (F.col("vulnerability_index") * 0.5 +
             F.coalesce(F.col("compound_index"), F.lit(0)) * 0.3 +
             F.col("dual_eligible_ratio") * 0.2) * 100) \
        .withColumn("impact_level",
            F.when(F.col("sdoh_impact_score") >= 75, "Critical Impact")
             .when(F.col("sdoh_impact_score") >= 50, "High Impact")
             .when(F.col("sdoh_impact_score") >= 25, "Moderate Impact")
             .otherwise("Low Impact")) \
        .select(
            gold_vulnerability_score.state,
            gold_vulnerability_score.county,
            F.round("sdoh_impact_score", 2).alias("sdoh_impact_score"),
            "impact_level",
            "vulnerability_index",
            "total_dual_eligible",
            F.current_timestamp().alias("calculated_at")
        )
    
    print(f"  Total regions: {metric_sdoh_impact.count()}")
    metric_sdoh_impact.orderBy(F.desc("sdoh_impact_score")).show(10, truncate=False)
    
    print("\nImpact Level Distribution:")
    metric_sdoh_impact.groupBy("impact_level").count().orderBy("impact_level").show()
else:
    print("  ✗ SDOH Impact metric unavailable - missing dual enrollment data")
    metric_sdoh_impact = None


METRIC 3/4: Social Determinants Impact
  Total regions: 125


25/12/02 22:56:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:56:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+-------+-----------------+---------------+-------------------+-------------------+--------------------------+
|state|county |sdoh_impact_score|impact_level   |vulnerability_index|total_dual_eligible|calculated_at             |
+-----+-------+-----------------+---------------+-------------------+-------------------+--------------------------+
|AL   |Coosa  |NaN              |Critical Impact|NaN                |4809               |2025-12-02 22:56:59.279256|
|AL   |SUMTER |24.12            |Low Impact     |0.2972416797277652 |8188               |2025-12-02 22:56:59.279256|
|AL   |Sumter |23.94            |Low Impact     |0.29356354220612196|8636               |2025-12-02 22:56:59.279256|
|AL   |GREENE |22.63            |Low Impact     |0.2794387281470948 |5689               |2025-12-02 22:56:59.279256|
|AL   |DALLAS |22.45            |Low Impact     |0.27756228939210537|35858              |2025-12-02 22:56:59.279256|
|AL   |MARENGO|22.38            |Low Impact     |0.2809412065122

25/12/02 22:57:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+---------------+-----+
|   impact_level|count|
+---------------+-----+
|Critical Impact|    1|
|     Low Impact|  124|
+---------------+-----+



25/12/02 22:57:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [30]:
print("\n" + "="*60)
print("METRIC 4/4: Geographic Disparity Magnitude")
print("="*60)

metric_disparity_magnitude = gold_regional_inequity \
    .join(gold_need_supply_gap, ["state", "zip", "county"], "left") \
    .join(gold_integrated_ranking, ["state", "zip", "county"], "left") \
    .withColumn("disparity_magnitude",
        (F.col("inequity_magnitude") * 0.4 +
         F.abs(F.col("gap_magnitude")) * 0.3 +
         F.col("national_percentile") * 0.3) * 100) \
    .withColumn("disparity_severity",
        F.when(F.col("disparity_magnitude") >= 80, "Extreme Disparity")
         .when(F.col("disparity_magnitude") >= 60, "High Disparity")
         .when(F.col("disparity_magnitude") >= 40, "Moderate Disparity")
         .otherwise("Low Disparity")) \
    .select(
        "state", "zip", "county",
        F.round("disparity_magnitude", 2).alias("geographic_disparity_magnitude"),
        "disparity_severity",
        "tier",
        "access_deviation",
        F.current_timestamp().alias("calculated_at")
    )

print(f"  Total regions: {metric_disparity_magnitude.count()}")
metric_disparity_magnitude.orderBy(F.desc("geographic_disparity_magnitude")).show(10, truncate=False)

print("\nDisparity Severity Distribution:")
metric_disparity_magnitude.groupBy("disparity_severity").count().orderBy("disparity_severity").show()

print("\n✓ All 4 final metrics calculated")


METRIC 4/4: Geographic Disparity Magnitude


  Total regions: 191


25/12/02 22:57:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----+---+------+------------------------------+------------------+-------+----------------+--------------------------+
|state|zip|county|geographic_disparity_magnitude|disparity_severity|tier   |access_deviation|calculated_at             |
+-----+---+------+------------------------------+------------------+-------+----------------+--------------------------+
|CT   |NaN|7040  |NULL                          |Low Disparity     |Top 20%|0.0             |2025-12-02 22:57:04.326802|
|AZ   |NaN|3060  |NULL                          |Low Disparity     |Top 20%|0.0             |2025-12-02 22:57:04.326802|
|CO   |NaN|6000  |NULL                          |Low Disparity     |Top 20%|0.0             |2025-12-02 22:57:04.326802|
|FL   |NaN|10450 |NULL                          |Low Disparity     |Top 20%|0.0             |2025-12-02 22:57:04.326802|
|54   |NaN|54070 |NULL                          |Low Disparity     |Top 20%|0.0             |2025-12-02 22:57:04.326802|
|54   |NaN|54999 |NULL          

25/12/02 22:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+------------------+-----+
|disparity_severity|count|
+------------------+-----+
|     Low Disparity|  191|
+------------------+-----+


✓ All 4 final metrics calculated


25/12/02 22:57:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


# STEP 6: Build DAG (Bronze -> Silver -> Gold -> Metrics)

In [31]:
# ============================================================================
# STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3) - Geographic Health Equity
# ============================================================================
# Features:
# - withColumn: extracts ALL columns (including chained operations)
# - agg metrics: extracts F.sum, F.count, F.avg, F.countDistinct, etc.
# - join: detects join operations with on/how parameters (supports multi-column joins)
# - filter: detects filter operations with conditions
# - select: detects select operations
# - Composite operations: join+agg+withColumn, outer joins, etc.
# - Handles intermediate DataFrames (person_location, condition_regional, etc.)
# - Supports 4-layer architecture: Bronze → Silver → Gold → Metric
# ============================================================================

print("\n" + "="*80)
print("STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)")
print("="*80)

import json
import re
import networkx as nx
from datetime import datetime
from typing import List, Dict, Tuple

# ============================================================================
# CONFIGURATION - Update these paths as needed
# ============================================================================
NOTEBOOK_FILE = "./3_geographic_health_equity.ipynb"

# Use existing LOCAL_DATA_DIR or set default
try:
    LOCAL_DATA_DIR
except NameError:
    LOCAL_DATA_DIR = "./3_data"
    import os
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# ============================================================================
# NODE DESCRIPTIONS - 3번 노트북 테이블 설명
# ============================================================================
NODE_DESCRIPTIONS = {
    # Bronze Layer (11 tables)
    "bronze_person": "OMOP: Person demographics with location_id",
    "bronze_location": "OMOP: Geographic location data (state, zip, county)",
    "bronze_care_site": "OMOP: Healthcare facility locations",
    "bronze_provider": "OMOP: Healthcare provider information",
    "bronze_condition": "OMOP: Condition occurrence records",
    "bronze_procedure": "OMOP: Procedure occurrence records",
    "bronze_drug": "OMOP: Drug exposure records",
    "bronze_obs_period": "OMOP: Patient observation periods",
    "bronze_dual_enroll": "SDOH: Dual-eligible enrollment by county",
    "bronze_hospital_info": "Medicare: Hospital general information",
    "bronze_inpatient": "Medicare: Inpatient charges 2011",
    
    # Intermediate DataFrames (not in final DAG but part of lineage)
    "person_location": "Intermediate: Person with location details",
    "provider_with_location": "Intermediate: Provider with care site and location",
    "condition_regional": "Intermediate: Conditions aggregated by region",
    "procedure_regional": "Intermediate: Procedures aggregated by region",
    "drug_regional": "Intermediate: Drug events aggregated by region",
    "base_disadvantage": "Intermediate: Base disadvantage combining underserved + efficiency",
    
    # Silver Layer (6 tables)
    "silver_regional_demographics": "Regional population demographics profile",
    "silver_provider_density": "Provider density by geographic region",
    "silver_service_utilization": "Regional healthcare service utilization",
    "silver_health_outcomes": "Regional health outcomes and disease burden",
    "silver_sdoh_indicators": "Social determinants of health indicators",
    "silver_facility_access": "Healthcare facility access metrics",
    
    # Gold Layer (15 tables)
    "gold_population_health_index": "Population health burden and diversity score",
    "gold_access_ratio": "Provider-to-population access ratio",
    "gold_utilization_intensity": "Service utilization intensity per capita",
    "gold_vulnerability_score": "Population vulnerability index (SDOH-based)",
    "gold_quality_access": "Quality-adjusted healthcare access score",
    "gold_service_diversity": "Healthcare service diversity index",
    "gold_care_accessibility": "Multi-dimensional care accessibility score",
    "gold_regional_inequity": "Regional healthcare inequity metrics",
    "gold_underserved_flag": "Underserved area identification flags",
    "gold_utilization_efficiency": "Healthcare utilization efficiency score",
    "gold_need_supply_gap": "Healthcare need vs supply gap analysis",
    "gold_compound_disadvantage": "Compound disadvantage index",
    "gold_temporal_stability": "Temporal stability assessment",
    "gold_integrated_ranking": "Integrated regional ranking",
    "gold_predictive_risk": "Predictive deterioration risk score",
    
    # Metric Layer (4 final KPIs)
    "metric_health_equity_index": "FINAL: Health Equity Index by Region",
    "metric_care_desert": "FINAL: Care Desert Score",
    "metric_sdoh_impact": "FINAL: Social Determinants Impact Score",
    "metric_disparity_magnitude": "FINAL: Geographic Disparity Magnitude",
}

# ============================================================================
# LINEAGE PARSER CLASS
# ============================================================================
class LineageParserV3:
    """
    Improved PySpark Lineage Parser for Geographic Health Equity Pipeline
    - Parses notebook code cells to extract DataFrame transformations
    - Supports: withColumn, groupBy+agg, join (multi-column), filter, select, fillna
    - Handles intermediate DataFrames
    - Outputs edge-centric lineage format for RAG retrieval
    """
    
    def __init__(self):
        self.edges = []
        self.G = nx.DiGraph()
    
    def get_layer(self, node_id: str) -> str:
        """Extract layer from node ID (bronze/silver/gold/metric)"""
        node_lower = node_id.lower()
        if 'bronze' in node_lower:
            return 'bronze'
        elif 'silver' in node_lower:
            return 'silver'
        elif 'gold' in node_lower:
            return 'gold'
        elif 'metric' in node_lower:
            return 'metric'
        return 'intermediate'
    
    def extract_all_withcolumns(self, code_block: str) -> List[str]:
        """Extract all column names from withColumn operations"""
        columns = []
        normalized = re.sub(r'\s+', ' ', code_block)
        pattern = r'\.withColumn\s*\(\s*["\']([^"\']+)["\']'
        for match in re.finditer(pattern, normalized):
            col = match.group(1)
            if col not in columns:
                columns.append(col)
        return columns
    
    def extract_agg_metrics(self, agg_block: str) -> Dict[str, str]:
        """Extract all metrics from agg block (F.sum, F.count, etc.)"""
        metrics = {}
        normalized = re.sub(r'\s+', ' ', agg_block)
        
        # Various patterns for F.func(...).alias(...)
        patterns = [
            r'F\s*\.\s*(\w+)\s*\(\s*"([^"]*)"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)',
            r"F\s*\.\s*(\w+)\s*\(\s*'([^']*)'\s*\)\s*\.\s*alias\s*\(\s*'([^']+)'\s*\)",
            r'F\s*\.\s*(\w+)\s*\(\s*"\*"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)',
        ]
        
        for pattern in patterns[:2]:
            for match in re.finditer(pattern, normalized):
                func = match.group(1)
                col = match.group(2)
                alias = match.group(3)
                metrics[alias] = f"{func}({col})"
        
        # Handle F.count("*").alias("name")
        for match in re.finditer(patterns[2], normalized):
            func = match.group(1)
            alias = match.group(2)
            metrics[alias] = f"{func}(*)"
        
        return metrics
    
    def extract_groupby_cols(self, groupby_str: str) -> List[str]:
        """Extract column names from groupBy clause"""
        cols = []
        for match in re.finditer(r'["\']([^"\']+)["\']', groupby_str):
            cols.append(match.group(1))
        return cols
    
    def extract_filter_condition(self, code_block: str) -> str:
        """Extract filter condition"""
        pattern = r'\.filter\s*\(\s*([^)]+)\s*\)'
        match = re.search(pattern, code_block)
        if match:
            cond = match.group(1).strip()
            cond = re.sub(r'F\.col\s*\(\s*["\']([^"\']+)["\']\s*\)', r'\1', cond)
            return cond[:100]
        return None
    
    def extract_join_info(self, code_block: str) -> List[Dict[str, str]]:
        """Extract ALL join information (supports multiple joins and multi-column joins)"""
        joins = []
        normalized = re.sub(r'\s+', ' ', code_block)
        
        # Pattern 1: .join(df, condition, "type")
        pattern_complex = r'\.join\s*\(\s*(\w+)(?:\.alias\s*\(\s*["\'](\w+)["\']\s*\))?\s*,\s*([^,]+)\s*,\s*["\'](\w+)["\']\s*\)'
        
        for match in re.finditer(pattern_complex, normalized):
            other = match.group(1)
            alias = match.group(2) if match.group(2) else None
            on_clause = match.group(3).strip()
            how = match.group(4)
            
            # Handle multi-column join: ["col1", "col2", "col3"]
            if '[' in on_clause:
                cols = re.findall(r'["\']([^"\']+)["\']', on_clause)
                on_col = cols if cols else on_clause[:30]
            else:
                col_match = re.search(r'["\']([^"\']+)["\']', on_clause)
                on_col = col_match.group(1) if col_match else on_clause[:30]
            
            joins.append({
                "other": alias if alias else other,
                "on": on_col,
                "how": how
            })
        
        # Pattern 2: .join(df, df.col == df2.col, "type")
        pattern_eq = r'\.join\s*\(\s*(\w+)\s*,\s*(\w+\.\w+)\s*==\s*(\w+\.\w+)\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_eq, normalized):
                other = match.group(1)
                on_col = f"{match.group(2)} == {match.group(3)}"
                how = match.group(4)
                joins.append({"other": other, "on": on_col, "how": how})
        
        # Pattern 3: Simple .join(df, "col", "how")
        pattern_simple = r'\.join\s*\(\s*(\w+)\s*,\s*["\']([^"\']+)["\']\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_simple, normalized):
                joins.append({
                    "other": match.group(1),
                    "on": match.group(2),
                    "how": match.group(3)
                })
        
        return joins if joins else None
    
    def extract_select_cols(self, code_block: str) -> List[str]:
        """Extract columns from select operation"""
        cols = []
        pattern = r'\.select\s*\(\s*([^)]+)\s*\)'
        match = re.search(pattern, code_block)
        if match:
            select_str = match.group(1)
            for col_match in re.finditer(r'["\']([^"\']+)["\']', select_str):
                cols.append(col_match.group(1))
        return cols
    
    def extract_fillna_cols(self, code_block: str) -> List[str]:
        """Extract columns from fillna operation"""
        pattern = r'\.fillna\s*\([^,]+,\s*subset\s*=\s*\[([^\]]+)\]'
        match = re.search(pattern, code_block)
        if match:
            cols_str = match.group(1)
            cols = re.findall(r'["\']([^"\']+)["\']', cols_str)
            return cols
        # Simple fillna(0) without subset
        if '.fillna(0)' in code_block or '.fillna(' in code_block:
            return ['all_columns']
        return []
    
    def parse_assignment(self, code: str) -> List[Dict]:
        """Parse DataFrame assignment statements from code"""
        edges = []
        lines = code.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            assign_match = re.match(r'(\w+)\s*=\s*\(?\s*(\w+)?', line)
            if assign_match and '=' in line and not line.startswith('#'):
                target = assign_match.group(1)
                
                # Skip non-DataFrame variables
                skip = {'spark', 'print', 'if', 'for', 'while', 'def', 'class', 
                       'client', 'query', 'output_file', 'stats', 'rag_data', 
                       'dag_file', 'G', 'lineage_edges', 'edges', 'node_attrs',
                       'label', 'layer', 'in_degree', 'out_degree', 'parents',
                       'children', 'texts', 'tables_to_download'}
                if target.lower() in skip or target in skip:
                    i += 1
                    continue
                
                # Collect full statement
                full_statement = line
                paren_count = line.count('(') - line.count(')')
                backslash_continue = line.rstrip().endswith('\\')
                j = i + 1
                
                while (paren_count > 0 or backslash_continue) and j < len(lines):
                    next_line = lines[j]
                    full_statement += '\n' + next_line
                    paren_count += next_line.count('(') - next_line.count(')')
                    backslash_continue = next_line.rstrip().endswith('\\')
                    j += 1
                
                # Process bronze/silver/gold/metric or intermediate DataFrames
                keywords = ['bronze', 'silver', 'gold', 'metric', 'person_location', 
                           'provider_with_location', 'condition_regional', 'procedure_regional',
                           'drug_regional', 'base_disadvantage']
                if any(x in full_statement.lower() for x in keywords):
                    edge = self.parse_single_assignment(target, full_statement)
                    if edge:
                        edges.append(edge)
                
                i = j
            else:
                i += 1
        
        return edges
    
    def parse_single_assignment(self, target: str, full_statement: str) -> Dict:
        """Parse a single DataFrame assignment statement"""
        # Find source DataFrame
        source_match = re.search(r'=\s*\(?\s*(\w+)(?:\s*\\)?\s*\.', full_statement)
        if not source_match:
            return None
        
        source = source_match.group(1)
        
        # Skip non-DataFrame sources
        if source.lower() in {'spark', 'f', 'w', 'os', 'pd', 'nx', 'json', 'print'}:
            return None
        
        sources = [source]
        op_types = []
        
        # Parse joins (can be multiple)
        joins_info = self.extract_join_info(full_statement)
        if joins_info:
            for join in joins_info:
                if join['other'] not in sources:
                    sources.append(join['other'])
            op_types.append('join')
        
        # Parse filter
        filter_cond = self.extract_filter_condition(full_statement)
        if filter_cond:
            op_types.append('filter')
        
        # Parse groupBy
        groupby_match = re.search(r'\.groupBy\s*\(\s*([^)]+)\s*\)', full_statement)
        groupby_cols = []
        if groupby_match:
            groupby_cols = self.extract_groupby_cols(groupby_match.group(1))
            op_types.append('agg')
        
        # Parse agg metrics
        metrics = {}
        agg_start = full_statement.find('.agg(')
        if agg_start != -1:
            paren_count = 0
            content_start = agg_start + 5
            content_end = content_start
            
            for idx, char in enumerate(full_statement[agg_start:]):
                if char == '(':
                    paren_count += 1
                elif char == ')':
                    paren_count -= 1
                    if paren_count == 0:
                        content_end = agg_start + idx
                        break
            
            agg_content = full_statement[content_start:content_end]
            metrics = self.extract_agg_metrics(agg_content)
        
        # Parse withColumn
        columns = self.extract_all_withcolumns(full_statement)
        if columns:
            op_types.append('withColumn')
        
        # Parse select
        select_cols = self.extract_select_cols(full_statement)
        if select_cols and not op_types:
            op_types.append('select')
        
        # Parse fillna
        fillna_cols = self.extract_fillna_cols(full_statement)
        if fillna_cols:
            op_types.append('fillna')
        
        # Skip if no operations found
        if not op_types:
            return None
        
        op_str = '+'.join(op_types)
        
        # Build operation dict
        operation = {"op": op_str}
        if joins_info:
            operation["joins"] = joins_info
        if filter_cond:
            operation["filter"] = filter_cond
        if groupby_cols:
            operation["groupBy"] = groupby_cols
        if metrics:
            operation["metrics"] = metrics
        if columns:
            operation["columns"] = columns
        if select_cols:
            operation["select"] = select_cols
        if fillna_cols:
            operation["fillna_columns"] = fillna_cols
        
        # Generate description text
        text_parts = [f"{target} is created from {source}:"]
        if joins_info:
            for j in joins_info:
                on_str = j['on'] if isinstance(j['on'], str) else ', '.join(j['on'])
                text_parts.append(f"{j['how']}-joins with {j['other']} on [{on_str}]")
        if filter_cond:
            text_parts.append(f"filters by {filter_cond[:50]}")
        if groupby_cols:
            text_parts.append(f"groups by {groupby_cols}")
        if metrics:
            text_parts.append(f"computes {list(metrics.keys())}")
        if columns:
            text_parts.append(f"adds columns {columns}")
        if select_cols:
            text_parts.append(f"selects {select_cols}")
        if fillna_cols:
            text_parts.append(f"fills nulls")
        
        text = " ".join(text_parts) + "."
        
        return {
            "id": f"{' + '.join(sources)} -> {target} ({op_str})",
            "source_nodes": sources,
            "target_node": target,
            "operation": operation,
            "text": text
        }
    
    def parse_notebook(self, notebook_path: str) -> Tuple[List[Dict], nx.DiGraph]:
        """Parse entire notebook file and extract lineage"""
        print(f"\n📖 Reading notebook: {notebook_path}")
        
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
        
        # Extract code cells
        code_cells = []
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code':
                source = cell.get('source', [])
                code = ''.join(source) if isinstance(source, list) else source
                code_cells.append(code)
        
        print(f"   Found {len(code_cells)} code cells")
        
        # Parse each cell
        all_edges = []
        for code in code_cells:
            edges = self.parse_assignment(code)
            all_edges.extend(edges)
        
        # Deduplicate by target (keep most complete edge)
        seen_targets = {}
        for edge in all_edges:
            target = edge['target_node']
            if target not in seen_targets:
                seen_targets[target] = edge
            else:
                existing = seen_targets[target]
                existing_score = len(existing.get('operation', {}).get('columns', [])) + \
                                len(existing.get('operation', {}).get('metrics', {})) + \
                                len(existing['source_nodes'])
                new_score = len(edge.get('operation', {}).get('columns', [])) + \
                           len(edge.get('operation', {}).get('metrics', {})) + \
                           len(edge['source_nodes'])
                if new_score > existing_score:
                    seen_targets[target] = edge
        
        self.edges = list(seen_targets.values())
        
        # Build graph
        self.build_graph()
        
        print(f"   Extracted {len(self.edges)} transformation edges")
        
        return self.edges, self.G
    
    def build_graph(self):
        """Build NetworkX DAG from edges"""
        self.G = nx.DiGraph()
        
        for edge in self.edges:
            target = edge['target_node']
            if not self.G.has_node(target):
                self.G.add_node(target, id=target, label=target, layer=self.get_layer(target))
            
            for src in edge['source_nodes']:
                if not self.G.has_node(src):
                    self.G.add_node(src, id=src, label=src, layer=self.get_layer(src))
                self.G.add_edge(src, target, etype="consume")
    
    def generate_rag_data(self) -> List[Dict]:
        """Generate RAG-compatible node data"""
        rag_data = []
        
        for node_id in self.G.nodes():
            layer = self.get_layer(node_id)
            in_deg = self.G.in_degree(node_id)
            out_deg = self.G.out_degree(node_id)
            parents = list(self.G.predecessors(node_id))
            children = list(self.G.successors(node_id))
            
            description = NODE_DESCRIPTIONS.get(node_id, node_id)
            
            texts = [
                description,
                f"Layer: {layer}",
                f"Incoming edges: {in_deg}, Outgoing edges: {out_deg}"
            ]
            if parents:
                texts.append(f"Consumes: {', '.join(parents)}")
            if children:
                texts.append(f"Feeds into: {', '.join(children)}")
            
            rag_data.append({"id": node_id, "texts": texts})
        
        return rag_data


# ============================================================================
# MAIN EXECUTION
# ============================================================================
print("\n" + "="*60)
print("Parsing Notebook for PySpark Transformations")
print("="*60)

parser = LineageParserV3()

try:
    edges, G = parser.parse_notebook(NOTEBOOK_FILE)
    rag_data = parser.generate_rag_data()
    
    # Statistics
    bronze_count = len([n for n in G.nodes() if parser.get_layer(n) == 'bronze'])
    silver_count = len([n for n in G.nodes() if parser.get_layer(n) == 'silver'])
    gold_count = len([n for n in G.nodes() if parser.get_layer(n) == 'gold'])
    metric_count = len([n for n in G.nodes() if parser.get_layer(n) == 'metric'])
    intermediate_count = len([n for n in G.nodes() if parser.get_layer(n) == 'intermediate'])
    
    print("\n" + "="*60)
    print("Parsing Results")
    print("="*60)
    print(f"  Total nodes: {G.number_of_nodes()}")
    print(f"  Total graph edges: {G.number_of_edges()}")
    print(f"  Lineage records: {len(edges)}")
    print(f"  Bronze: {bronze_count}, Silver: {silver_count}, Gold: {gold_count}, Metric: {metric_count}")
    print(f"  Intermediate: {intermediate_count}")
    print(f"  Is DAG: {nx.is_directed_acyclic_graph(G)}")
    
    # ========================================================================
    # SAVE OUTPUTS
    # ========================================================================
    print("\n" + "="*60)
    print("Saving Outputs")
    print("="*60)
    
    lineage_file = f"{LOCAL_DATA_DIR}/geographic_health_equity_lineage_edges_auto.json"
    with open(lineage_file, 'w', encoding='utf-8') as f:
        json.dump(edges, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved lineage: {lineage_file}")
    
    rag_file = f"{LOCAL_DATA_DIR}/geographic_health_equity_rag_data_auto.json"
    with open(rag_file, 'w', encoding='utf-8') as f:
        json.dump(rag_data, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved RAG data: {rag_file}")
    
    dag_file = f"{LOCAL_DATA_DIR}/geographic_health_equity_dag_auto.graphml"
    nx.write_graphml(G, dag_file)
    print(f"✔ Saved DAG: {dag_file}")
    
    stats = {
        "pipeline": "Geographic Health Equity (Auto-Parsed v3)",
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "lineage_edges": len(edges),
        "bronze_nodes": bronze_count,
        "silver_nodes": silver_count,
        "gold_nodes": gold_count,
        "metric_nodes": metric_count,
        "intermediate_nodes": intermediate_count,
        "is_dag": nx.is_directed_acyclic_graph(G),
        "timestamp": datetime.now().isoformat(),
        "source_notebook": NOTEBOOK_FILE
    }
    
    stats_file = f"{LOCAL_DATA_DIR}/dag_statistics_auto.json"
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved statistics: {stats_file}")
    
    # ========================================================================
    # DISPLAY RESULTS
    # ========================================================================
    print("\n" + "="*80)
    print("AUTO-PARSING COMPLETE (v3)")
    print("="*80)
    
    print("\n--- Extracted Transformations ---\n")
    for i, edge in enumerate(edges, 1):
        print(f"{i}. [{edge['operation']['op']}] {edge['id']}")
        print(f"   Sources: {edge['source_nodes']}")
        print(f"   Target:  {edge['target_node']}")
        
        op = edge['operation']
        if 'columns' in op and op['columns']:
            print(f"   Columns: {op['columns']}")
        if 'groupBy' in op and op['groupBy']:
            print(f"   GroupBy: {op['groupBy']}")
        if 'metrics' in op and op['metrics']:
            print(f"   Metrics: {op['metrics']}")
        if 'joins' in op:
            for j in op['joins']:
                on_str = j['on'] if isinstance(j['on'], str) else ', '.join(j['on'])
                print(f"   Join:    {j['how']} on [{on_str}] with {j['other']}")
        if 'filter' in op:
            print(f"   Filter:  {op['filter']}")
        if 'select' in op:
            print(f"   Select:  {op['select']}")
        print(f"   Text:    {edge['text']}")
        print()
    
    print("="*80)
    print(f"Total: {len(edges)} transformations, {G.number_of_nodes()} nodes")
    print(f"Files saved to: {LOCAL_DATA_DIR}/")
    print("="*80)

except FileNotFoundError:
    print(f"\n❌ Error: Notebook file not found: {NOTEBOOK_FILE}")
    print("   Update NOTEBOOK_FILE path at the top of this cell.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()


STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)

Parsing Notebook for PySpark Transformations

📖 Reading notebook: ./3_geographic_health_equity.ipynb
   Found 35 code cells
   Extracted 33 transformation edges

Parsing Results
  Total nodes: 44
  Total graph edges: 42
  Lineage records: 33
  Bronze: 9, Silver: 6, Gold: 15, Metric: 4
  Intermediate: 10
  Is DAG: True

Saving Outputs
✔ Saved lineage: ./3_data/geographic_health_equity_lineage_edges_auto.json
✔ Saved RAG data: ./3_data/geographic_health_equity_rag_data_auto.json
✔ Saved DAG: ./3_data/geographic_health_equity_dag_auto.graphml
✔ Saved statistics: ./3_data/dag_statistics_auto.json

AUTO-PARSING COMPLETE (v3)

--- Extracted Transformations ---

1. [join+agg+withColumn] bronze_person + bronze_location -> silver_regional_demographics (join+agg+withColumn)
   Sources: ['bronze_person', 'bronze_location']
   Target:  silver_regional_demographics
   Columns: ['processed_at']
   GroupBy: ['state', 'zip', 'county']
   Metrics: {'

# SUMMARY

In [40]:
print("\n" + "="*80)
print("GEOGRAPHIC HEALTH EQUITY PIPELINE COMPLETE")
print("="*80)

print(f"\nData Sources:")
print(f"  - OMOP Clinical: 8 tables")
print(f"  - Medicare: 2 tables")
print(f"  - Dual Enrollment SDOH: 1 table")

print(f"\nPipeline Architecture:")
print(f"  - Bronze Layer: {len(bronze_nodes)} tables (raw geographic & clinical data)")
print(f"  - Silver Layer: {len(silver_nodes)} tables (regional aggregations)")
print(f"  - Gold Layer: {len(gold_nodes)} tables (vertical transformations)")
print(f"  - Metrics Layer: {len(metric_nodes)} final KPIs")

print(f"\nFinal Metrics:")
print(f"  1. Health Equity Index by Region")
print(f"  2. Care Desert Score")
print(f"  3. Social Determinants Impact")
print(f"  4. Geographic Disparity Magnitude")

print(f"\nDAG:")
print(f"  - Total transformations: {G.number_of_nodes()} nodes")
print(f"  - Data dependencies: {G.number_of_edges()} edges")
print(f"  - Valid DAG: {nx.is_directed_acyclic_graph(G)}")

print(f"\nCost Optimization:")
print(f"  - Strategy: LIMIT {LIMIT} on all BigQuery reads")
print(f"  - Storage: Local CSV (Pandas -> Spark)")

print("\n" + "="*80)
print("✓ Ready for Marquez lineage tracking integration")
print("✓ Ready for regional health equity analysis")
print("="*80)


GEOGRAPHIC HEALTH EQUITY PIPELINE COMPLETE

Data Sources:
  - OMOP Clinical: 8 tables
  - Medicare: 2 tables
  - Dual Enrollment SDOH: 1 table

Pipeline Architecture:
  - Bronze Layer: 11 tables (raw geographic & clinical data)
  - Silver Layer: 6 tables (regional aggregations)
  - Gold Layer: 15 tables (vertical transformations)
  - Metrics Layer: 4 final KPIs

Final Metrics:
  1. Health Equity Index by Region
  2. Care Desert Score
  3. Social Determinants Impact
  4. Geographic Disparity Magnitude

DAG:
  - Total transformations: 36 nodes
  - Data dependencies: 54 edges
  - Valid DAG: True

Cost Optimization:
  - Strategy: LIMIT 1000 on all BigQuery reads
  - Storage: Local CSV (Pandas -> Spark)

✓ Ready for Marquez lineage tracking integration
✓ Ready for regional health equity analysis
